In [2]:
%%writefile bfs_benchmark.cu
#include <iostream>
#include <vector>
#include <queue>
#include <limits>
#include <cuda_runtime.h>

#define CHECK_CUDA(call)                                                      \
    do {                                                                      \
        cudaError_t err = call;                                               \
        if (err != cudaSuccess) {                                             \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err)            \
                      << " at " << __FILE__ << ":" << __LINE__ << std::endl;  \
            exit(EXIT_FAILURE);                                               \
        }                                                                     \
    } while (0)

constexpr int INF = -1;

// Parallel BFS Kernel: Vertex-centric frontier expansion using atomic operations
__global__ void bfs_kernel(const int* __restrict__ row_ptr,
                           const int* __restrict__ col_indices,
                           int* __restrict__ dist,
                           int* __restrict__ d_changed,
                           int current_level,
                           int num_vertices) {
    int u = blockIdx.x * blockDim.x + threadIdx.x;

    if (u < num_vertices && dist[u] == current_level) {
        int start = row_ptr[u];
        int end = row_ptr[u + 1];

        for (int edge = start; edge < end; ++edge) {
            int v = col_indices[edge];

            // Atomic Compare-And-Swap prevents race conditions when multiple
            // threads try to update the distance of the same neighbor vertex
            if (atomicCAS(&dist[v], INF, current_level + 1) == INF) {
                *d_changed = 1;
            }
        }
    }
}

// Sequential CPU BFS for baseline correctness validation
std::vector<int> cpu_bfs(int num_vertices,
                         const std::vector<int>& row_ptr,
                         const std::vector<int>& col_indices,
                         int source) {
    std::vector<int> dist(num_vertices, INF);
    std::queue<int> q;

    dist[source] = 0;
    q.push(source);

    while (!q.empty()) {
        int u = q.front();
        q.pop();

        for (int i = row_ptr[u]; i < row_ptr[u + 1]; ++i) {
            int v = col_indices[i];
            if (dist[v] == INF) {
                dist[v] = dist[u] + 1;
                q.push(v);
            }
        }
    }
    return dist;
}

int main() {
    // Generate a synthetic directed graph
    const int num_vertices = 1000000; // 1 Million vertices
    const int edges_per_node = 16;
    const int total_edges = num_vertices * edges_per_node;
    const int source_vertex = 0;

    std::cout << "Graph: " << num_vertices << " vertices, " << total_edges << " edges (CSR format)\n";

    std::vector<int> h_row_ptr(num_vertices + 1);
    std::vector<int> h_col_indices(total_edges);

    h_row_ptr[0] = 0;
    for (int i = 0; i < num_vertices; ++i) {
        h_row_ptr[i + 1] = h_row_ptr[i] + edges_per_node;
        for (int j = 0; j < edges_per_node; ++j) {
            // Pseudo-random connected topology
            h_col_indices[i * edges_per_node + j] = (i + j * 997 + 1) % num_vertices;
        }
    }

    // Host distances
    std::vector<int> h_dist(num_vertices, INF);
    h_dist[source_vertex] = 0;

    // Device allocations
    int *d_row_ptr, *d_col_indices, *d_dist, *d_changed;
    CHECK_CUDA(cudaMalloc(&d_row_ptr, (num_vertices + 1) * sizeof(int)));
    CHECK_CUDA(cudaMalloc(&d_col_indices, total_edges * sizeof(int)));
    CHECK_CUDA(cudaMalloc(&d_dist, num_vertices * sizeof(int)));
    CHECK_CUDA(cudaMalloc(&d_changed, sizeof(int)));

    CHECK_CUDA(cudaMemcpy(d_row_ptr, h_row_ptr.data(), (num_vertices + 1) * sizeof(int), cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaMemcpy(d_col_indices, h_col_indices.data(), total_edges * sizeof(int), cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaMemcpy(d_dist, h_dist.data(), num_vertices * sizeof(int), cudaMemcpyHostToDevice));

    int block_size = 256;
    int grid_size = (num_vertices + block_size - 1) / block_size;

    cudaEvent_t start, stop;
    CHECK_CUDA(cudaEventCreate(&start));
    CHECK_CUDA(cudaEventCreate(&stop));

    // Warm-up kernel to avoid driver initialization overhead in timing
    int level = 0;
    int h_changed = 1;
    CHECK_CUDA(cudaMemset(d_changed, 0, sizeof(int)));
    bfs_kernel<<<grid_size, block_size>>>(d_row_ptr, d_col_indices, d_dist, d_changed, level, num_vertices);
    CHECK_CUDA(cudaGetLastError());
    CHECK_CUDA(cudaDeviceSynchronize());

    // Reset for actual benchmark
    CHECK_CUDA(cudaMemcpy(d_dist, h_dist.data(), num_vertices * sizeof(int), cudaMemcpyHostToDevice));

    CHECK_CUDA(cudaEventRecord(start));

    // Iterative level expansion
    while (h_changed) {
        h_changed = 0;
        CHECK_CUDA(cudaMemset(d_changed, 0, sizeof(int)));

        bfs_kernel<<<grid_size, block_size>>>(d_row_ptr, d_col_indices, d_dist, d_changed, level, num_vertices);
        CHECK_CUDA(cudaGetLastError());
        CHECK_CUDA(cudaDeviceSynchronize());

        CHECK_CUDA(cudaMemcpy(&h_changed, d_changed, sizeof(int), cudaMemcpyDeviceToHost));
        level++;
    }

    CHECK_CUDA(cudaEventRecord(stop));
    CHECK_CUDA(cudaEventSynchronize(stop));

    float gpu_time = 0.0f;
    CHECK_CUDA(cudaEventElapsedTime(&gpu_time, start, stop));
    std::cout << "[GPU BFS] Completed in " << level << " levels.\n";
    std::cout << "[GPU BFS] Execution Time: " << gpu_time << " ms\n";
    std::cout << "Throughput: " << (total_edges / (gpu_time * 1e-3) / 1e6) << " MTEPS (Million Traversed Edges/s)\n";

    // Verify against CPU sequential BFS
    CHECK_CUDA(cudaMemcpy(h_dist.data(), d_dist, num_vertices * sizeof(int), cudaMemcpyDeviceToHost));
    std::vector<int> cpu_dist = cpu_bfs(num_vertices, h_row_ptr, h_col_indices, source_vertex);

    int mismatch_count = 0;
    for (int i = 0; i < num_vertices; ++i) {
        if (h_dist[i] != cpu_dist[i]) {
            mismatch_count++;
            if (mismatch_count <= 5) {
                std::cout << "Mismatch at vertex " << i << ": GPU=" << h_dist[i] << ", CPU=" << cpu_dist[i] << "\n";
            }
        }
    }

    std::cout << "Validation: " << (mismatch_count == 0 ? "SUCCESS (All vertex levels match)" : "FAILED")
              << " | Mismatches: " << mismatch_count << "\n";

    // Cleanup
    CHECK_CUDA(cudaFree(d_row_ptr));
    CHECK_CUDA(cudaFree(d_col_indices));
    CHECK_CUDA(cudaFree(d_dist));
    CHECK_CUDA(cudaFree(d_changed));
    CHECK_CUDA(cudaEventDestroy(start));
    CHECK_CUDA(cudaEventDestroy(stop));

    return mismatch_count == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}

Overwriting bfs_benchmark.cu


In [5]:
!ls -l bfs_benchmark.cu

-rw-r--r-- 1 root root 6679 Oct 16 19:42 bfs_benchmark.cu


In [6]:
!nvcc -O3 -arch=sm_75 bfs_benchmark.cu -o bfs_benchmark

In [7]:
!ls -lh bfs_benchmark

-rwxr-xr-x 1 root root 979K Oct 16 19:48 bfs_benchmark


In [8]:
!./bfs_benchmark

Graph: 1000000 vertices, 16000000 edges (CSR format)
[GPU BFS] Completed in 1064 levels.
[GPU BFS] Execution Time: 56.1702 ms
Throughput: 284.849 MTEPS (Million Traversed Edges/s)
Validation: SUCCESS (All vertex levels match) | Mismatches: 0
